# Dreamer: Agent Quality Distillation Pipeline

Nightly job that:
1. Goes through all child branches of the `production` Lakebase branch
2. For each branch, fetches the last 24 hours of chat logs
3. Anonymizes conversations (strips user IDs) and sends them to an LLM
4. Extracts generalized, anonymous insights on how the agent can improve
5. Synthesizes all per-session insights into a final report
6. Writes the report as a markdown file to a UC Volume for agent developers to review

Note: In your Job Task, make sure to configure your environment such that it includes the following dependencies:
1. databricks-langchain[memory]>=0.17.0
2. databricks-sdk>=0.94.0

In [ ]:
import json
import asyncio
from datetime import datetime

from databricks.sdk import WorkspaceClient
from databricks_langchain import ChatDatabricks
from databricks_ai_bridge.lakebase import LakebaseClient
from langchain_core.messages import SystemMessage, HumanMessage

## 1. Configuration

In [ ]:
# -- Lakebase project & branch config --
LAKEBASE_PROJECT = "memories-agent" # TODO: Update lakebase project name here
PRODUCTION_BRANCH = "production"

# -- LLM config --
llm = ChatDatabricks(endpoint="databricks-claude-opus-4-7", max_tokens=4000)

# -- Output config (persisted to UC Volume) --
_log_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_DIR = "/Volumes/catalog/schema/dreamer_logs" # TODO: Update volume directory here
REPORT_FILE = f"{LOG_DIR}/dreamer_agent_distillation_{_log_ts}.md"

# -- Databricks SDK client --
w = WorkspaceClient()

## 2. Observability Logger

In [ ]:
distillation_log = []


def log_event(event_type: str, branch: str = "", detail: str = ""):
    """Append a structured log entry to the in-memory log."""
    ts = datetime.now().isoformat()
    entry = f"[{ts}] [{event_type}] branch={branch} | {detail}"
    distillation_log.append(entry)
    print(entry)

## 3. Discover Child Branches of Production

In [ ]:
def list_child_branches(project: str, parent_branch: str) -> list:
    """
    List all branches in the project whose source (parent) is the given parent_branch.
    """
    parent_resource = f"projects/{project}"
    production_resource = f"projects/{project}/branches/{parent_branch}"

    all_branches = list(w.postgres.list_branches(parent=parent_resource))

    child_branches = []
    for branch in all_branches:
        if branch.name == production_resource:
            continue
        status = branch.status if hasattr(branch, "status") else None
        if status is None:
            continue
        source = (
            getattr(status, "source_branch", None)
            if not isinstance(status, dict)
            else status.get("source_branch")
        )
        if source == production_resource:
            child_branches.append(branch)

    return child_branches


def branch_id_from_name(branch_name: str) -> str:
    """Extract the branch_id from a full resource name like 'projects/foo/branches/bar'."""
    return branch_name.rsplit("/", 1)[-1]

## 4. Extract & Anonymize 24-Hour Logs from a Branch

In [ ]:
def _fetch_daily_chat_logs_sync(project: str, branch: str) -> list:
    """
    Query the last 24 hours of chat messages, returning anonymized
    conversation transcripts (one per user session). User IDs are
    replaced with opaque labels like 'User 1', 'User 2', etc.
    """
    with LakebaseClient(project=project, branch=branch) as client:
        query = """
            SELECT c."userId", m."chatId", m.role, m.parts
            FROM ai_chatbot."Message" m
            JOIN ai_chatbot."Chat" c ON m."chatId" = c.id
            WHERE m."createdAt" >= NOW() - INTERVAL '1 day'
            ORDER BY c."userId", m."chatId", m."createdAt";
        """
        rows = client.execute(query) or []

    # Group by user, then anonymize
    user_logs = {}
    for row in rows:
        user_id = str(row["userId"])
        if user_id not in user_logs:
            user_logs[user_id] = []
        user_logs[user_id].append(f"{row['role'].upper()}: {json.dumps(row['parts'])}")

    # Return anonymized transcripts as a list (no user IDs leak out)
    anonymized = []
    for i, (_, transcripts) in enumerate(user_logs.items(), 1):
        anonymized.append({
            "session_label": f"Session {i}",
            "message_count": len(transcripts),
            "transcript": "\n".join(transcripts),
        })

    return anonymized


async def fetch_daily_chat_logs(project: str, branch: str) -> list:
    """Run the sync LakebaseClient query in a thread to avoid blocking the async event loop."""
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(None, _fetch_daily_chat_logs_sync, project, branch)

## 5. Per-Session Insight Extraction

In [ ]:
SESSION_PROMPT = """
You are an Agent Quality Analyst. You are reviewing an anonymized chat session
between a user and an AI agent. Your job is to extract generalized insights that
would help an agent developer improve the agent's quality.

Focus on:
1. **Corrections**: Did the user correct the agent? What pattern was wrong?
2. **Repeated failures**: Did the agent fail at the same kind of task multiple times?
3. **Workarounds**: Did the user have to work around agent limitations?
4. **Confusion**: Were there misunderstandings between the user and agent?
5. **Quality wins**: Did the agent do something particularly well that should be reinforced?

Rules:
- Do NOT include any user-identifiable information (names, emails, IDs, specific project names).
- Generalize specific examples into reusable patterns.
- If the conversation is routine with no notable patterns, respond with "No notable insights."

Respond with a JSON object:
{
  "has_insights": true,
  "insights": [
    {
      "category": "correction | repeated_failure | workaround | confusion | quality_win",
      "summary": "One-sentence description of the pattern",
      "detail": "2-3 sentences explaining what happened and what should change",
      "severity": "low | medium | high"
    }
  ]
}
"""


async def extract_session_insights(session: dict, branch_name: str) -> list:
    """Send one anonymized session to the LLM and extract quality insights."""
    label = session["session_label"]
    log_event("ANALYZING_SESSION", branch_name, detail=f"{label} ({session['message_count']} messages)")

    messages = [
        SystemMessage(content=SESSION_PROMPT),
        HumanMessage(content=f"Anonymized Chat Transcript ({label}):\n{session['transcript']}"),
    ]

    try:
        response = llm.invoke(messages)
        raw = response.content.strip().replace("```json", "").replace("```", "")
        result = json.loads(raw)

        if result.get("has_insights") and result.get("insights"):
            insights = result["insights"]
            log_event("INSIGHTS_FOUND", branch_name, detail=f"{label}: {len(insights)} insights")
            return insights
        else:
            log_event("NO_INSIGHTS", branch_name, detail=f"{label}: no notable patterns")
            return []

    except Exception as e:
        log_event("SESSION_ERROR", branch_name, detail=f"{label}: {e}")
        return []

## 6. Final Synthesis Across All Sessions

In [ ]:
SYNTHESIS_PROMPT = """
You are an Agent Quality Lead producing a daily improvement report.

Below is a list of individual insights extracted from anonymized agent sessions
over the past 24 hours. Your job is to:

1. **Deduplicate**: Merge similar insights into single findings.
2. **Rank**: Order by frequency and severity (most impactful first).
3. **Categorize**: Group into these sections:
   - Procedural improvements (how the agent should behave differently)
   - Knowledge gaps (topics or domains where the agent struggles)
   - UX / interaction patterns (formatting, tone, verbosity issues)
   - Quality wins (things working well that should be preserved)
4. **Recommend**: For each finding, suggest a concrete action the agent developer can take.

Rules:
- Keep everything anonymous — no user data, IDs, or specific project details.
- Be concise but actionable.
- If there are very few or no meaningful patterns, say so honestly.

Respond in clean markdown (no code fences around the whole response).
"""


async def synthesize_insights(all_insights: list, stats: dict) -> str:
    """Take all per-session insights and produce a final synthesized report."""
    if not all_insights:
        return "No notable insights were found across any sessions in the past 24 hours."

    insights_text = json.dumps(all_insights, indent=2)

    messages = [
        SystemMessage(content=SYNTHESIS_PROMPT),
        HumanMessage(
            content=(
                f"Summary stats: {stats['total_sessions']} sessions analyzed "
                f"across {stats['total_branches']} branches, "
                f"{stats['total_messages']} total messages.\n\n"
                f"Individual insights:\n{insights_text}"
            )
        ),
    ]

    response = llm.invoke(messages)
    return response.content.strip()

## 7. Main Pipeline

In [ ]:
async def run_agent_distillation_pipeline():
    log_event("PIPELINE_START", PRODUCTION_BRANCH, detail="Discovering child branches...")

    child_branches = list_child_branches(LAKEBASE_PROJECT, PRODUCTION_BRANCH)
    branch_names = [branch_id_from_name(b.name) for b in child_branches]
    log_event(
        "BRANCHES_FOUND",
        PRODUCTION_BRANCH,
        detail=f"{len(child_branches)} child branches: {branch_names}",
    )

    if not child_branches:
        log_event("PIPELINE_END", PRODUCTION_BRANCH, detail="No child branches found.")
        return

    all_insights = []
    total_sessions = 0
    total_messages = 0

    for branch in child_branches:
        branch_name = branch_id_from_name(branch.name)
        log_event("BRANCH_START", branch_name, detail="Fetching chat logs...")

        try:
            sessions = await fetch_daily_chat_logs(LAKEBASE_PROJECT, branch_name)
            log_event("BRANCH_LOGS", branch_name, detail=f"{len(sessions)} anonymized sessions")

            for session in sessions:
                total_sessions += 1
                total_messages += session["message_count"]
                insights = await extract_session_insights(session, branch_name)
                all_insights.extend(insights)

            log_event("BRANCH_DONE", branch_name, detail="Complete.")

        except Exception as e:
            log_event("BRANCH_ERROR", branch_name, detail=str(e))

    # Synthesize all insights into a final report
    log_event(
        "SYNTHESIS_START",
        detail=f"Synthesizing {len(all_insights)} raw insights from {total_sessions} sessions...",
    )

    stats = {
        "total_branches": len(child_branches),
        "total_sessions": total_sessions,
        "total_messages": total_messages,
        "raw_insights_count": len(all_insights),
    }

    synthesized_report = await synthesize_insights(all_insights, stats)
    log_event("SYNTHESIS_DONE", detail="Final report generated.")

    # Build the full markdown report
    report_lines = [
        f"# Agent Quality Distillation Report",
        f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        "",
        "| Metric | Value |",
        "|--------|-------|",
        f"| Branches scanned | {stats['total_branches']} |",
        f"| Sessions analyzed | {stats['total_sessions']} |",
        f"| Total messages | {stats['total_messages']} |",
        f"| Raw insights extracted | {stats['raw_insights_count']} |",
        "",
        "---",
        "",
        synthesized_report,
        "",
        "---",
        "",
        "## Raw Insights (Pre-Synthesis)",
        "",
        "<details>",
        "<summary>Click to expand all raw insights</summary>",
        "",
        "```json",
        json.dumps(all_insights, indent=2),
        "```",
        "",
        "</details>",
        "",
        "---",
        "",
        "## Pipeline Log",
        "",
        "```",
    ] + distillation_log + ["```"]

    report_md = "\n".join(report_lines)

    # Write to UC Volume
    with open(REPORT_FILE, "w") as f:
        f.write(report_md)

    log_event("PIPELINE_END", detail=f"Report written to {REPORT_FILE}")
    return report_md

## 8. Run the Pipeline

In [ ]:
report = await run_agent_distillation_pipeline()

## 9. View the Report

In [ ]:
# Build clickable URL to view the file in the Databricks workspace UI
def _make_volume_url(file_path: str) -> str:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    workspace_id = ctx.workspaceId().get()
    host = w.config.host.rstrip("/")
    # file_path is like /Volumes/<catalog>/<schema>/<volume>/<filename>
    parts = file_path.removeprefix("/Volumes/").split("/", 3)
    volume_dir = "/".join(parts[:3])
    file_name = parts[3] if len(parts) > 3 else ""
    return f"{host}/explore/data/volumes/{volume_dir}?o={workspace_id}&filePreviewPath={file_name}"


if report:
    print(report)
    print(f"\nReport saved to: {REPORT_FILE}")
    print(f"View report at: {_make_volume_url(REPORT_FILE)}")
else:
    print("No report generated (no child branches or no sessions found).")